In [2]:
import pandas as pd
import numpy as np

import joblib

from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import importlib
import sys

sys.path.append("../src")

import preprocessing

importlib.reload(preprocessing)

from preprocessing import Preprocessing

In [3]:
df = pd.read_csv("../datasets/processed/traffic_processed.csv")

prep = Preprocessing(df)

df = prep.remove_unused_columns()

X, y = prep.split_features_target()

X_train, X_test, y_train, y_test = prep.train_test(X, y)

preprocessor = prep.create_preprocessor()

In [4]:
models = {

    "Decision Tree":
        DecisionTreeRegressor(
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            random_state=42
        )
}

In [5]:
results = []

for name, model in models.items():

    pipe = Pipeline([

        ("preprocessor", preprocessor),

        ("model", model)

    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = mean_squared_error(
        y_test,
        pred
    ) ** 0.5

    r2 = r2_score(
        y_test,
        pred
    )

    results.append([

        name,
        mae,
        rmse,
        r2

    ])

In [8]:
results = pd.DataFrame(

    results,

    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R² Score"
    ]

)

results = results.sort_values(
    by="R² Score",
    ascending=False
).reset_index(drop=True)

results

,Model,MAE,RMSE,R² Score
0,Gradient Boosting,2.885949,4.284945,0.966059
1,Random Forest,2.848689,4.328610,0.965364
2,Extra Trees,2.878684,4.417253,0.963931
3,Decision Tree,3.632835,5.962177,0.934289


In [9]:
best = results.iloc[0]

print(best)

Model       Gradient Boosting
MAE                  2.885949
RMSE                 4.284945
R² Score             0.966059
Name: 0, dtype: object
